# 04 - Deploy the React Application

**Business Context:** HydraB needs a modern web application that combines fleet location, battery health, delivery pipeline, and CRM data in one place. The app is hosted entirely within Snowflake using Snowpark Container Services (SPCS).

**What this notebook does:**
- Deploys a pre-built React/Next.js application as a container on Snowflake's compute pool
- The app reads directly from the GOLD schema - no data copies, no external hosting, no credentials to manage
- Provides a public HTTPS endpoint accessible to any authenticated Snowflake user

**Why this matters for HydraB:**
- **Modern application** - React/Next.js gives you a full web framework (not a basic dashboard). Custom UI, interactive maps, multi-page navigation.
- **Governance built in** - Snowflake RBAC (role-based access control) applies to the app. Users only see data their role permits.
- **AI and ML ready from day 1** - The app can call Cortex AI functions, embed the fleet agent, or surface ML predictions directly in the UI. No separate infrastructure needed.
- **No data egress** - Everything stays inside Snowflake's security perimeter. No external servers, no API keys, no data leaving the platform.


## Set Session Context
Resolves the per-user database name and sets the active warehouse.

In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;
USE WAREHOUSE HYDRAB_HOL_WH;

## Create SPCS Service

**Snowpark Container Services (SPCS)** lets you run custom containers directly inside Snowflake - no external hosting, no data egress, no credentials to manage.

To get to this point, the admin setup already completed these steps:
1. **Built the React app** into a Docker image (Next.js + Recharts + Leaflet)
   - Important: must build with `--platform linux/amd64` since SPCS runs x86_64 nodes
2. **Pushed the image** to Snowflake's built-in container registry (`IMAGE_REPO`)
3. **Created a Compute Pool** (`HYDRAB_HOL_POOL`) - a cluster of nodes that runs containers

Now we just deploy it. The service reads directly from our GOLD schema - no API layer, no data copies.

Documentation: [Snowpark Container Services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview)


In [ ]:
-- Create the SPCS service from pre-built image
CREATE SERVICE IF NOT EXISTS DASHBOARD_SERVICE
  IN COMPUTE POOL HYDRAB_HOL_POOL
  FROM SPECIFICATION $$
  spec:
    containers:
    - name: dashboard
      image: /HYDRAB_HOL_SHARED/PUBLIC/IMAGE_REPO/hydrab-dashboard:v8
      env:
        SNOWFLAKE_DATABASE: HYDRAB_HOL_NHUVAERE
        HOSTNAME: "0.0.0.0"
      resources:
        requests:
          cpu: 0.5
          memory: 512M
        limits:
          cpu: 1
          memory: 1G
    endpoints:
    - name: dashboard
      port: 3000
      public: true
  $$
  MIN_INSTANCES = 1
  MAX_INSTANCES = 1;

## Check Service Status
Polls the service status. Re-run until it shows `RUNNING` (~1-2 minutes on first deploy).

In [ ]:
-- Wait for service to start (check status)
-- Run this cell a few times until status = RUNNING
SELECT SYSTEM$GET_SERVICE_STATUS('GOLD.DASHBOARD_SERVICE');

## Get Public URL
Shows the ingress URL for the application. Open this in your browser.

In [ ]:
-- Get the public endpoint URL
SHOW ENDPOINTS IN SERVICE GOLD.DASHBOARD_SERVICE;

## Application Deployed!

Copy the `ingress_url` from above and open it in your browser.
You should see the HydraB Vehicle 360 application with:
- Overview stats (vehicles, pipeline, deliveries)
- Fleet Map with GPS pins
- Sales Pipeline funnel chart
- Vehicle telemetry detail pages

---

## What's Next: Extend with Cortex Code Desktop

The `react-app/` folder in this skill contains the full source code.
Open it in Cortex Code Desktop and ask CoCo to:
- Add a new page (e.g. Delivery Tracking)
- Connect charts to live Snowflake queries
- Improve the Fleet Map with vehicle popups
- Add the Cortex Agent chat to the application

In [ ]:
-- Cleanup (run when done with the lab)
-- DROP SERVICE IF EXISTS GOLD.DASHBOARD_SERVICE;
-- DROP DATABASE IF EXISTS IDENTIFIER($user_db);